# local_MPC

Run and cache compare-ready Local MPC rollouts for both perfect and LSTM forecasts so `compare.ipynb` can replay them without re-solving.

Local MPC uses an `economic_only` objective. If you need SOC soft constraints, use the global MPC or ADMM MPC notebooks instead.


In [ ]:
from pathlib import Path
import importlib
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

repo_root = Path.cwd().resolve()
while repo_root != repo_root.parent and not (repo_root / "configs").exists():
    repo_root = repo_root.parent
if not (repo_root / "configs").exists():
    raise RuntimeError("Could not locate the project root from the notebook working directory.")
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import configs as configs_pkg
from configs.profiles import compose_experiment_config
from scripts.mainline_compare import LOCAL_MPC_LSTM_LABEL, LOCAL_MPC_PERFECT_LABEL, collect_local_mpc_rollout, compare_rollout_metrics
from scripts.utils import grid_notebook_workflow as grid_nb
from scripts.utils.project_paths import project_root as resolve_project_root

configs_pkg = importlib.reload(configs_pkg)
grid_nb = importlib.reload(grid_nb)


In [ ]:
PROJECT_ROOT = resolve_project_root()
DATA_DIR = PROJECT_ROOT / "data"

TEST_START_DATE = "2020-06-01"
TEST_END_DATE = "2020-06-04"
BASE_PREDICTION_MODE = "normal"
LOAD_SCALE = [10.0] * 5
PV_SCALE = [5.0] * 5
BATTERY_CAPACITY_KWH = 20.0
BATTERY_MAX_POWER_KW = 10.0
BATTERY_MAX_CHARGE_RATE = BATTERY_MAX_POWER_KW / BATTERY_CAPACITY_KWH
BATTERY_CONTROLS = {"battery_capacity": BATTERY_CAPACITY_KWH, "max_charge_rate": BATTERY_MAX_CHARGE_RATE}


In [ ]:
base_cfg = compose_experiment_config(
    profile="base",
    algorithm="MATD3",
    model_family="mlp",
    data_dir=DATA_DIR,
    runtime_mode="performance",
)
grid_nb.apply_notebook_experiment_settings(
    base_cfg,
    prediction_mode=BASE_PREDICTION_MODE,
    test_start_date=TEST_START_DATE,
    test_end_date=TEST_END_DATE,
    load_scale=LOAD_SCALE,
    pv_scale=PV_SCALE,
    battery_controls=BATTERY_CONTROLS,
)
normal_comparison_cfg = grid_nb.build_comparison_cfg(base_cfg, prediction_mode="normal")
normal_comparison_cfg.runtime.forecast_ready = grid_nb.ensure_forecast_ready(normal_comparison_cfg)

display(
    pd.Series(
        {
            "test_start_date": base_cfg.data.test_start_date,
            "test_end_date": base_cfg.data.test_end_date,
            "base_prediction_mode": BASE_PREDICTION_MODE,
            "normal_forecast_backend": normal_comparison_cfg.forecast.type,
            "future_horizon": base_cfg.env.future_horizon,
            "episode_limit": base_cfg.env.episode_limit,
            "local_mpc_objective_mode": "economic_only",
            "import_price_markup_eur_per_kwh": float(base_cfg.reward.import_price_markup_eur_per_kwh),
            "export_subsidy_eur_per_kwh": float(base_cfg.reward.export_subsidy_eur_per_kwh),
            "normal_forecast_ready": normal_comparison_cfg.runtime.forecast_ready is not None,
        },
        name="local_mpc_notebook_config",
    )
)


In [ ]:
print(f"Starting {LOCAL_MPC_PERFECT_LABEL}...")
local_mpc_perfect = collect_local_mpc_rollout(
    base_cfg,
    prediction_mode="perfect",
    label=LOCAL_MPC_PERFECT_LABEL,
)
print("Done:", local_mpc_perfect.meta.get("controller", LOCAL_MPC_PERFECT_LABEL))

print(f"Starting {LOCAL_MPC_LSTM_LABEL}...")
local_mpc_lstm = collect_local_mpc_rollout(
    base_cfg,
    prediction_mode="normal",
    label=LOCAL_MPC_LSTM_LABEL,
)
print("Done:", local_mpc_lstm.meta.get("controller", LOCAL_MPC_LSTM_LABEL))


In [ ]:
metrics_df = compare_rollout_metrics(local_mpc_perfect, local_mpc_lstm)
display(metrics_df)


display(
    pd.Series(
        {
            "local_mpc_perfect_controller": local_mpc_perfect.meta.get("controller"),
            "local_mpc_lstm_controller": local_mpc_lstm.meta.get("controller"),
            "local_mpc_perfect_forecast_backend": local_mpc_perfect.meta.get("forecast_backend"),
            "local_mpc_lstm_forecast_backend": local_mpc_lstm.meta.get("forecast_backend"),
        },
        name="local_mpc_rollout_summary",
    )
)


In [ ]:
grid_nb.plot_price_prediction_comparison(local_mpc_perfect, local_mpc_lstm)
plt.show()

grid_nb.plot_net_load_comparison(local_mpc_perfect, local_mpc_lstm)
plt.show()

grid_nb.plot_battery_power_and_soc_comparison(local_mpc_perfect, local_mpc_lstm)
plt.show()
